In [1]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image, ImageOps, ImageEnhance

# Seeds
random.seed(21)
torch.manual_seed(21)

# Paths
dataset_path = "/home/justin/HTR"
images_path = os.path.join(dataset_path, "self_lines")
labels_path = os.path.join(dataset_path, "lines.txt")

# Parse labels
samples = []
with open(labels_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        filename = parts[0] + ".png"
        text = " ".join(parts[9:]).replace("|", " ")
        image_path = os.path.join(images_path, filename)
        if os.path.exists(image_path):
            samples.append({"image_path": image_path, "text": text})

# Split
random.shuffle(samples)
train_size = int(0.70 * len(samples))
val_size   = int(0.15 * len(samples))
train_samples = samples[:train_size]
val_samples   = samples[train_size:train_size + val_size]
test_samples  = samples[train_size + val_size:]

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/justin/HTR\\lines.txt'

In [ ]:
class HandwritingDataset(Dataset):
    def __init__(self, samples, processor, augment=False, max_target_length=128):
        self.samples = samples
        self.processor = processor
        self.augment = augment
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)

        if self.augment:
            angle = random.uniform(-3, 3)
            image = image.rotate(angle, fillcolor=255)
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))

        image = image.convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample["text"],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

train_dataset = HandwritingDataset(train_samples, processor, augment=True)
val_dataset   = HandwritingDataset(val_samples,   processor, augment=False)
test_dataset  = HandwritingDataset(test_samples,  processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

print("DataLoaders complete")

In [ ]:
from torch.optim import AdamW
from jiwer import cer, wer
from transformers import GenerationConfig

def evaluate(model, processor, test_samples, device):
    model.eval()
    predictions = []
    ground_truths = []

    for sample in test_samples:
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)
        image = image.convert("RGB")
        
        #new, stores widths
        widths.append(image.size[0])

        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device).float()

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                generation_config=model.generation_config,
            )

        predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(predicted_text)
        ground_truths.append(sample["text"])

    #new, compiles results 
    results = {
        "cer": cer(ground_truths, predictions),
        "wer": wer(ground_truths, predictions),
        "predictions": predictions,
        "ground_truths": ground_truths,
        "widths": widths,
    }

    #new, returns compiled results
    return results


def train(learning_rate, num_epochs, augment=True, label="experiment"):
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print(f"\n{'='*50}")
    print(f"Experiment: {label}")
    print(f"LR: {learning_rate} | Epochs: {num_epochs} | Augment: {augment}")
    print(f"Device: {device}")
    print(f"{'='*50}")

    model = VisionEncoderDecoderModel.from_pretrained(
        "microsoft/trocr-small-handwritten",
        torch_dtype=torch.float32
    )

    model.config.decoder_start_token_id = 1
    model.config.pad_token_id = 1
    model.config.eos_token_id = 2

    model.generation_config = GenerationConfig(
        decoder_start_token_id=1,
        eos_token_id=2,
        pad_token_id=1,
        max_new_tokens=64,
        forced_eos_token_id=None,
        forced_bos_token_id=None,
    )

    model.decoder.config.decoder_start_token_id = 1
    model.decoder.config.eos_token_id = 2

    model.to(device)
    model = model.float()

    optimizer = AdamW(model.parameters(), lr=learning_rate)

    t_dataset = HandwritingDataset(train_samples, processor, augment=augment)
    t_loader  = DataLoader(t_dataset, batch_size=4, shuffle=True)

    history = {
        "train_loss": [],
        "val_cer": [],
        "val_wer": []
    }

    print("Starting training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in t_loader:
            batch_pixels = batch["pixel_values"].to(device).float()
            batch_labels = batch["labels"].to(device)

            outputs = model(pixel_values=batch_pixels, labels=batch_labels)
            loss    = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(t_loader)

        #new, since the return of evaluate(...) is a dictionary now
        val_results = evaluate(model, processor, val_samples, device)
        val_cer = val_results["cer"]
        val_wer = val_results["wer"]

        history["train_loss"].append(avg_loss)
        history["val_cer"].append(val_cer)
        history["val_wer"].append(val_wer)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Val CER: {val_cer:.2%} | Val WER: {val_wer:.2%}")

    #new, since the return of evaluate(...) is a dictionary now
    test_results = evaluate(model, processor, test_samples, device)
    test_cer = test_results["cer"]
    test_wer = test_results["wer"]
    print(f"\nFinal Test CER: {test_cer:.2%} | Test WER: {test_wer:.2%}")

    #new, since the return of evaluate(...) is a dictionary now
    return model, history, test_results

print("Loaded training and evaluation")

In [ ]:
model_1, history_1, test_results_1 = train(
    learning_rate=5e-5, num_epochs=5, augment=True, label="lr=5e-5, epochs=5, augment=True"
)

In [ ]:
model_2, history_2, test_results_2 = train(
    learning_rate=1e-5, num_epochs=5, augment=True, label="lr=1e-5, epochs=5, augment=True"
)

In [ ]:
model_3, history_3, test_results_3 = train(
    learning_rate=1e-4, num_epochs=5, augment=True, label="lr=1e-4, epochs=5, augment=True"
)

In [ ]:
model_4, history_4, test_results_4 = train(
    learning_rate=5e-5, num_epochs=5, augment=False, label="lr=5e-5, epochs=5, augment=False"
)

## Findings

### Models were trained with 5 epochs

### Model 1
Learning Rate: 5e-5
Augmented: True
CER: 92.34%
WER: 112.30%

### Model 2
Learning Rate: 1e-5
Augmented: True
CER: 12.76%
WER: 28.69%

### Model 3
Learning Rate: 1e-4
Augmented: True
CER: 81.67%
WER: 98.36%

### Model 4
Learning Rate: 1e-5
Augmented: True
CER: 71.18%
WER: 91.26%

### Results
Model 2 had the lowest overall error rates

In [ ]:
model_best, history_best, cer_best, wer_best = train(
    learning_rate=1e-5, num_epochs=15, augment=True, label="lr=1e-5, epochs=15, augment=True"
)

## Further training
The best modal was then further trained with 15 epochs

### Results
Epochs: 15
Learning Rate: 1e-5
Augmented: True
CER: 6.77%
WER: 17.49%

This model was a big improvement compared to the baseline.

CER: 13.69% vs 6.77%
WER: 48.63% vs 17.49%

In [3]:
def save_ocr_training_run_summary(
    history,
    test_results,
    learning_rate,
    num_epochs,
    augment,
    label,
    train_count,
    val_count,
    test_count,
    results_dir="results",
    dataset_name="HTR/self_lines",
    model_name="microsoft/trocr-small-handwritten",
    top_n_chars=10,
):
    import json
    import uuid
    import socket
    import getpass
    import subprocess
    from pathlib import Path
    from datetime import datetime
    from collections import defaultdict

    import numpy as np
    import pandas as pd
    from jiwer import cer

    def get_git_commit():
        try:
            return subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"],
                text=True
            ).strip()
        except Exception:
            return "unknown"

    #metadata
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    run_id = uuid.uuid4().hex[:8]
    user = getpass.getuser()
    hostname = socket.gethostname()
    git_commit = get_git_commit()

    predictions = test_results["predictions"]
    ground_truths = test_results["ground_truths"]
    widths = test_results["widths"]
    test_cer = float(test_results["cer"])
    test_wer = float(test_results["wer"])

    #per-sample CERs for width bins
    sample_cers = [
        cer([truth], [pred])
        for truth, pred in zip(ground_truths, predictions)
    ]

    short = [sample_cers[i] for i, w in enumerate(widths) if w < 1000]
    medium = [sample_cers[i] for i, w in enumerate(widths) if 1000 <= w < 2000]
    long_ = [sample_cers[i] for i, w in enumerate(widths) if w >= 2000]

    short_avg_cer = float(np.mean(short)) if short else None
    medium_avg_cer = float(np.mean(medium)) if medium else None
    long_avg_cer = float(np.mean(long_)) if long_ else None

    #character-level difficulty
    char_correct = defaultdict(int)
    char_total = defaultdict(int)

    for pred, truth in zip(predictions, ground_truths):
        for char in truth:
            char_total[char] += 1
            if char in pred:
                char_correct[char] += 1

    char_accuracy = {
        char: char_correct[char] / char_total[char]
        for char in char_total
        if char_total[char] >= 5
    }

    sorted_chars = sorted(char_accuracy.items(), key=lambda x: x[1])
    hardest_chars = sorted_chars[:top_n_chars]

    hardest_chars_data = [
        {
            "char": char,
            "accuracy": float(acc),
            "samples": int(char_total[char]),
        }
        for char, acc in hardest_chars
    ]

    hardest_chars_str = "; ".join(
        f"{repr(item['char'])}: {item['accuracy']:.2%} ({item['samples']})"
        for item in hardest_chars_data
    )

    #training history
    final_train_loss = float(history["train_loss"][-1]) if history["train_loss"] else None
    final_val_cer = float(history["val_cer"][-1]) if history["val_cer"] else None
    final_val_wer = float(history["val_wer"][-1]) if history["val_wer"] else None

    best_val_cer = float(min(history["val_cer"])) if history["val_cer"] else None
    best_val_wer = float(min(history["val_wer"])) if history["val_wer"] else None

    # -----------------------------
    # 5. Build one-row summary
    # -----------------------------
    summary_row = {
        "run_id": run_id,
        "timestamp": timestamp,
        "user": user,
        "hostname": hostname,
        "git_commit": git_commit,

        "label": label,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "augment": augment,

        "dataset_name": dataset_name,
        "model_name": model_name,

        "train_count": train_count,
        "val_count": val_count,
        "test_count": test_count,

        "test_cer": test_cer,
        "test_wer": test_wer,

        "final_train_loss": final_train_loss,
        "final_val_cer": final_val_cer,
        "final_val_wer": final_val_wer,
        "best_val_cer": best_val_cer,
        "best_val_wer": best_val_wer,

        "short_avg_cer": short_avg_cer,
        "short_count": len(short),
        "medium_avg_cer": medium_avg_cer,
        "medium_count": len(medium),
        "long_avg_cer": long_avg_cer,
        "long_count": len(long_),

        "hardest_chars_summary": hardest_chars_str,
        "hardest_chars_json": json.dumps(hardest_chars_data, ensure_ascii=False),
    }

    summary_df = pd.DataFrame([summary_row])

    # -----------------------------
    # 6. Save CSV
    # -----------------------------
    out_dir = Path(results_dir) / user
    out_dir.mkdir(parents=True, exist_ok=True)

    safe_label = "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in label)
    out_file = out_dir / f"{timestamp}_{run_id}_{safe_label}_summary.csv"

    summary_df.to_csv(out_file, index=False)

    # -----------------------------
    # 7. Print readable summary
    # -----------------------------
    print("\n===== TRAINING RUN SUMMARY =====")
    print(f"Run ID:          {run_id}")
    print(f"Timestamp:       {timestamp}")
    print(f"User:            {user}")
    print(f"Host:            {hostname}")
    print(f"Git commit:      {git_commit}")

    print(f"\nExperiment:      {label}")
    print(f"Learning rate:   {learning_rate}")
    print(f"Epochs:          {num_epochs}")
    print(f"Augment:         {augment}")

    print(f"\nDataset:         {dataset_name}")
    print(f"Model:           {model_name}")
    print(f"Train/Val/Test:  {train_count} / {val_count} / {test_count}")

    print(f"\nTest CER:        {test_cer:.2%}")
    print(f"Test WER:        {test_wer:.2%}")

    if final_train_loss is not None:
        print(f"\nFinal train loss: {final_train_loss:.4f}")
    if final_val_cer is not None:
        print(f"Final val CER:    {final_val_cer:.2%}")
    if final_val_wer is not None:
        print(f"Final val WER:    {final_val_wer:.2%}")
    if best_val_cer is not None:
        print(f"Best val CER:     {best_val_cer:.2%}")
    if best_val_wer is not None:
        print(f"Best val WER:     {best_val_wer:.2%}")

    if short_avg_cer is not None:
        print(f"\nShort images  — avg CER: {short_avg_cer:.2%} ({len(short)} samples)")
    else:
        print("\nShort images  — no samples")

    if medium_avg_cer is not None:
        print(f"Medium images — avg CER: {medium_avg_cer:.2%} ({len(medium)} samples)")
    else:
        print("Medium images — no samples")

    if long_avg_cer is not None:
        print(f"Long images   — avg CER: {long_avg_cer:.2%} ({len(long_)} samples)")
    else:
        print("Long images   — no samples")

    print(f"\nHardest characters (top {top_n_chars}):")
    for item in hardest_chars_data:
        print(f"  {repr(item['char'])}: {item['accuracy']:.2%} accuracy ({item['samples']} samples)")

    print(f"\nSaved summary CSV to: {out_file}")

    return summary_df, out_file

In [4]:
summary_df_1, out_file_1 = save_ocr_training_run_summary(
    history=history_1,
    test_results=test_results_1,
    learning_rate=5e-5,
    num_epochs=5,
    augment=True,
    label="lr=5e-5, epochs=5, augment=True",
    train_count=len(train_samples),
    val_count=len(val_samples),
    test_count=len(test_samples),
)

NameError: name 'history_1' is not defined

In [5]:
summary_df_2, out_file_2 = save_ocr_training_run_summary(
    history=history_2,
    test_results=test_results_2,
    learning_rate=1e-5,
    num_epochs=5,
    augment=True,
    label="lr=1e-5, epochs=5, augment=True",
    train_count=len(train_samples),
    val_count=len(val_samples),
    test_count=len(test_samples),
)

NameError: name 'history_2' is not defined

In [6]:
summary_df_3, out_file_3 = save_ocr_training_run_summary(
    history=history_3,
    test_results=test_results_3,
    learning_rate=1e-4,
    num_epochs=5,
    augment=True,
    label="lr=1e-4, epochs=5, augment=True",
    train_count=len(train_samples),
    val_count=len(val_samples),
    test_count=len(test_samples),
)

NameError: name 'history_3' is not defined

In [7]:
summary_df_4, out_file_4 = save_ocr_training_run_summary(
    history=history_4,
    test_results=test_results_4,
    learning_rate=5e-5,
    num_epochs=5,
    augment=False,
    label="lr=5e-5, epochs=5, augment=False",
    train_count=len(train_samples),
    val_count=len(val_samples),
    test_count=len(test_samples),
)

NameError: name 'history_4' is not defined